##Setup: Imports and Drive Connection

In [ ]:
# here we are importing all the required libraries
import tensorflow as tf
from tensorflow.keras.layers import Input, GlobalAveragePooling2D, Dense, Dropout, RandomFlip, RandomRotation
from tensorflow.keras.models import Model
from tensorflow.keras.applications import ResNet50
import matplotlib.pyplot as plt
import os
import numpy as np

# here we are connecting to google drive
print("connecting to google drive...")
from google.colab import drive
drive.mount('/content/drive')
print("drive mounted successfully")


Connecting to Google Drive...
Mounted at /content/drive
Drive mounted successfully.


## Load Data: Unzip and Define Paths

In [ ]:
# here we are unzipping the dataset from google drive
print("unzipping the data zip file from drive...")
ZIP_PATH = "/content/drive/MyDrive/DATA.zip"
!unzip -o -q {ZIP_PATH} -d "/content/"
print("data is unzipped and ready in content")

# here we are defining the main project variables
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 4
EPOCHS_STAGE_1 = 15
EPOCHS_STAGE_2 = 10

# here we are defining the paths to our training validation and testing folders
TRAIN_DIR = "/content/DATA/Training(70%)"
VALID_DIR = "/content/DATA/Validation(20%)"
TEST_DIR = "/content/DATA/Testing(10%)"


Unzipping the DATA.zip file from Drive...
Data is unzipped and ready in /content/.


## Create Data Pipelines

In [ ]:
# here we are loading the training validation and test datasets
print("loading training validation and test datasets...")
train_dataset = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR, label_mode="categorical", seed=123,
    image_size=IMAGE_SIZE, batch_size=BATCH_SIZE
)

# here we are loading the validation dataset
validation_dataset = tf.keras.utils.image_dataset_from_directory(
    VALID_DIR, label_mode="categorical", seed=123,
    image_size=IMAGE_SIZE, batch_size=BATCH_SIZE
)

# here we are loading the test dataset without shuffling
test_dataset = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR, label_mode="categorical", image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE, shuffle=False
)

# here we are checking the class names
class_names = train_dataset.class_names
print(f"our class names are: {class_names}")

# here we are optimizing the data pipelines for faster performance
print("optimizing data pipelines...")
AUTOTUNE = tf.data.AUTOTUNE
train_dataset = train_dataset.cache().prefetch(buffer_size=AUTOTUNE)
validation_dataset = validation_dataset.cache().prefetch(buffer_size=AUTOTUNE)
test_dataset = test_dataset.cache().prefetch(buffer_size=AUTOTUNE)


Loading Training, Validation, and Test datasets...
Found 2297 files belonging to 4 classes.
Found 573 files belonging to 4 classes.
Found 394 files belonging to 4 classes.
Our class names are: ['glioma_tumor', 'meningioma_tumor', 'no_tumor', 'pituitary_tumor']
Optimizing data pipelines...


## Stage 1: Initial Model Training (Head Only)

In [ ]:
# here we are building the resnet50 model
print("building the resnet50 model...")
base_model = ResNet50(weights='imagenet', include_top=False,
                      input_shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3))

# here we are starting stage 1 for feature extraction
print("freezing the resnet50 base layers for stage 1")
base_model.trainable = False

# here we are creating the model architecture with augmentation and classifier head
inputs = Input(shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3))
x = RandomFlip('horizontal')(inputs)
x = RandomRotation(0.1)(x)
x = base_model(x, training=False)
x = GlobalAveragePooling2D()(x)
x = Dropout(0.3)(x)
outputs = Dense(NUM_CLASSES, activation='softmax')(x)
model = Model(inputs, outputs)

# here we are compiling the model for stage 1
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# here we are starting the training for stage 1
print("starting model training stage 1 head only")
history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=EPOCHS_STAGE_1,
    verbose=1
)

# here we are finishing the training for stage 1
print("stage 1 training complete")


Building the ResNet50 model...
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step
Freezing the ResNet50 base layers for Stage 1.

--- Starting Model Training (Stage 1: Head Only) ---
Epoch 1/15
72/72 ━━━━━━━━━━━━━━━━━━━━ 29s 231ms/step - accuracy: 0.4653 - loss: 1.3200 - val_accuracy: 0.7609 - val_loss: 0.6219
Epoch 2/15
72/72 ━━━━━━━━━━━━━━━━━━━━ 11s 155ms/step - accuracy: 0.7378 - loss: 0.6368 - val_accuracy: 0.8307 - val_loss: 0.4805
Epoch 3/15
72/72 ━━━━━━━━━━━━━━━━━━━━ 11s 160ms/step - accuracy: 0.8001 - loss: 0.5224 - val_accuracy: 0.8482 - val_loss: 0.4386
Epoch 4/15
72/72 ━━━━━━━━━━━━━━━━━━━━ 12s 164ms/step - accuracy: 0.8231 - loss: 0.4519 - val_accuracy: 0.8360 - val_loss: 0.4519
Epoch 5/15
72/72 ━━━━━━━━━━━━━━━━━━━━ 12s 165ms/step - accuracy: 0.8290 - loss: 0.4263 - val_accuracy: 0.8534 - val_loss: 0.4241
Epoch 6/15
72/72 ━━━━━━━━━━━━━━━━━━━━ 12s 163ms/step - accuracy: 0.8518 - loss: 0.4002 - val_accuracy: 0.8709 - val_loss: 0.3671
Epoch 7/15
72/72 ━━━━━━━━━━━━━━━━━━━━ 11s 1

## Stage 2: Fine-Tuning (Advanced)

In [ ]:
# here we are starting stage 2 fine tuning
print("starting model training stage 2 fine tuning")
print("unfreezing the top 30 layers of the model...")

# here we are unfreezing the base model to allow fine tuning
base_model.trainable = True

# here we are keeping all layers frozen except the last 30
for layer in base_model.layers[:-30]:
    layer.trainable = False

# here we are recompiling the model for fine tuning with a smaller learning rate
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy',
             tf.keras.metrics.Precision(name='precision'),
             tf.keras.metrics.Recall(name='recall')]
)

# here we are confirming the model has been recompiled for fine tuning
print("model recompiled for fine tuning")
model.summary()

# here we are continuing the training to fine tune the unfrozen layers
print("continuing training to fine tune the unfrozen layers...")
history_finetune = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=EPOCHS_STAGE_2,
    initial_epoch=history.epoch[-1],
    verbose=1
)

# here we are finishing the fine tuning stage
print("stage 2 fine tuning complete")



--- Starting Model Training (Stage 2: Fine-Tuning) ---
Unfreezing the top 30 layers of the model...
--- Model Re-compiled for Fine-Tuning ---


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_flip (RandomFlip)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_rotation                 │ (None, 224, 224, 3)    │             0 │
│ (RandomRotation)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 4)              │         8,196 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,595,908 (90.01 MB)

 Trainable params: 14,458,372 (55.15 MB)

 Non-trainable params: 9,137,536 (34.86 MB)

Continuing training to fine-tune the unfrozen layers...
--- Stage 2 Fine-Tuning Complete ---


## Final Evaluation and Saving

In [ ]:
# here we are evaluating the fine tuned model on the test set
print("evaluating the fine tuned model on the test set")
results = model.evaluate(test_dataset, verbose=1)

# here we are collecting all the evaluation metrics
metrics = {}
metrics['loss'] = results[0]
metrics['accuracy'] = results[1]
metrics['precision'] = results[2]
metrics['recall'] = results[3]

# here we are calculating the f1 score manually
if (metrics['precision'] + metrics['recall']) > 0:
    metrics['f1_score'] = 2 * (metrics['precision'] * metrics['recall']) / (metrics['precision'] + metrics['recall'])
else:
    metrics['f1_score'] = 0.0

# here we are printing the test results
print("fine tuned model test results")
print(f"test loss: {metrics['loss']:.4f}")
print(f"test accuracy: {metrics['accuracy']:.4f}")
print(f"test precision: {metrics['precision']:.4f}")
print(f"test recall: {metrics['recall']:.4f}")
print(f"test f1 score: {metrics['f1_score']:.4f}")

# here we are saving the final model to google drive
print("saving the model to google drive")
os.makedirs("/content/drive/MyDrive/MODELS", exist_ok=True)
model.save("/content/drive/MyDrive/MODELS/resnet_finetuned.h5")

# here we are confirming that the fine tuned model has been saved successfully
print("fine tuned resnet model saved to drive")
print("we can now compare this result to our baseline")



--- Evaluating the Fine-Tuned Model on the Test Set ---
13/13 ━━━━━━━━━━━━━━━━━━━━ 6s 160ms/step - accuracy: 0.4293 - loss: 2.2956 - precision: 0.4288 - recall: 0.4009



--- Fine-Tuned Model Test Results ---
Test Loss: 1.3644
Test Accuracy: 0.6066
Test Precision: 0.6176
Test Recall: 0.5863
Test F1-Score: 0.6016

--- Saving the model to Google Drive ---
Fine-Tuned ResNet model saved to /content/drive/MyDrive/MODELS/resnet_finetuned.h5
We can now compare this to our baseline!
